# xHuBERT Experiment 4: Ablation Study
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

## Configurations
| Config | SLA | AP | Description |
|--------|-----|----|-------------|
| HuBERT-vanilla-FT | No | No | Fine-tune + last_hidden_state + mean-pool |
| xHuBERT-SLA | Yes | No | +Selective Layer Aggregation |
| xHuBERT-AP | No | Yes | +Attention Pooling |
| xHuBERT-full | Yes | Yes | Proposed method |

**Protocol**: LOSGO x 3 seeds x 6 folds = 72 runs (~72h total)

**Runtime**: **T4 GPU**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

WORK = "/content/drive/MyDrive/xhubert_results"
os.makedirs(WORK, exist_ok=True)

os.environ["XHUBERT_SAVE_DIR"] = WORK
os.environ["RAVDESS_ROOT"] = os.path.join(WORK, "RAVDESS")

print(f"Working directory: {WORK}")

In [ ]:
!pip install -q --upgrade transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
import os, subprocess

REPO_DIR = "/content/ravdess_experiment"

if os.path.exists(REPO_DIR):
    print("Repo exists, pulling latest ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("Cloning repo ...")
    subprocess.run([
        "git", "clone", "-b", "feature/xhubert-rewrite",
        "https://github.com/nhunet/ravdess_experiment.git", REPO_DIR
    ], check=True)

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

# Verify required files
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
    "experiments/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("All required files OK")

In [ ]:
import os

DRIVE_RAVDESS = os.environ["RAVDESS_ROOT"]

if not os.path.exists(DRIVE_RAVDESS):
    print("Not in Drive -> Download from Zenodo...")
    !wget -q --show-progress https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d /content/RAVDESS_tmp
    !mkdir -p "$DRIVE_RAVDESS"
    !cp -r /content/RAVDESS_tmp/* "$DRIVE_RAVDESS/"
    !rm -rf /content/RAVDESS_tmp Audio_Speech_Actors_01-24.zip
    print("Saved to Drive.")
else:
    import glob
    n = len(glob.glob(os.path.join(DRIVE_RAVDESS, "**/*.wav"), recursive=True))
    print(f"Already exist ({n} wav files), don't download.")

In [ ]:
import config
from utils import ensure_dirs

ensure_dirs()
print(f"SAVE_DIR:  {config.SAVE_DIR}")
print(f"CSV_DIR:   {config.CSV_DIR}")
print(f"FIG_DIR:   {config.FIG_DIR}")
print(f"CKPT_DIR:  {config.CKPT_DIR}")
print(f"EMB_DIR:   {config.EMB_DIR}")
print(f"LOG_DIR:   {config.LOG_DIR}")
print(f"RAVDESS:   {config.RAVDESS_ROOT}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Load Dataset

In [ ]:
from data import RavdessDataset
import config

dataset = RavdessDataset(sr=config.SR_HUBERT)
dataset.print_summary()

## Quick Test

In [ ]:
from experiments.exp4_ablation import run_exp4

df_quick = run_exp4(dataset=dataset, seeds=[42], configs=["HuBERT-vanilla-FT"],
                    force=True, quick=True)
print("Quick test passed!" if len(df_quick) > 0 else "FAILED!")

## Session 1: seed 42 (all 4 configs)

In [ ]:
df_s1 = run_exp4(dataset=dataset, seeds=[42], force=False)
print(df_s1.groupby("Model")["accuracy"].agg(["mean", "std"]).round(2))

## Session 2: seeds 43, 44

In [ ]:
df_s2 = run_exp4(dataset=dataset, seeds=[43, 44], force=False)
print(df_s2.groupby(["Model", "Seed"])["accuracy"].agg(["mean", "std"]).round(2))

## Statistical Tests

In [ ]:
import pandas as pd, numpy as np, os, config
from stats import paired_ttest, format_result

csv_path = os.path.join(config.CSV_DIR, "results_exp4_ablation.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    avg = df.groupby(["Model", "Fold"])["accuracy"].mean().reset_index()

    vanilla = avg[avg["Model"] == "HuBERT-vanilla-FT"]["accuracy"].values
    full = avg[avg["Model"] == "xHuBERT-full"]["accuracy"].values

    if len(vanilla) == len(full) and len(vanilla) > 0:
        print(format_result("xHuBERT-full", "HuBERT-vanilla-FT", "Accuracy", full, vanilla))
else:
    print("Run ablation first!")

## Visualization

In [ ]:
from visualization.plots import plot_exp4_ablation
import pandas as pd, os, config

csv_path = os.path.join(config.CSV_DIR, "results_exp4_ablation.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    plot_exp4_ablation(df)
    print("Figure saved!")
else:
    print("Run ablation first!")